# Export Performance Analysis

Analyze a **completed** Genesys Cloud resource export from **`TF_LOG=json`** trace output.

Expects export start/end lines from the resource exporter (`Started processing for resource`, `Collected resource`). Reports export counts, durations, and longest-running resources.

Capture example:

```bash
export TF_LOG=json
export TF_LOG_PATH=export-tflog.log
# optional: GENESYSCLOUD_SDK_DEBUG=true GENESYSCLOUD_SDK_DEBUG_FORMAT=Json
# run your Genesys Cloud resource export
export TERRAFORM_LOG_PATH=export-tflog.log
```

For hung or partial exports use `export/hang-analysis.ipynb`. Run `whatisit.ipynb` if unsure.

In [ ]:
import sys
from pathlib import Path

for _root in (Path.cwd(), *Path.cwd().parents):
    if (_root / "commonlib" / "config.py").is_file():
        _notebooks_root = _root
        break
    if (_root / "notebooks" / "commonlib" / "config.py").is_file():
        _notebooks_root = _root / "notebooks"
        break
else:
    raise RuntimeError(
        "Could not find notebooks/commonlib/. Start Jupyter from notebooks/ "
        "or open a notebook under export/, plan/, apply/, sdk-plan/, or the notebooks root."
    )

if str(_notebooks_root) not in sys.path:
    sys.path.insert(0, str(_notebooks_root))

from commonlib.notebook_setup import setup

setup()

import pandas as pd
import commonlib.prep_export_data as prep_export_data
import commonlib.gencharts as gencharts
import matplotlib.pyplot as plt
import commonlib.config as cfg


In [ ]:
c = cfg.Config()
print(c.TERRAFORM_LOG_PATH)
normalized_records = prep_export_data.load_normalized_records()
df = pd.json_normalize(normalized_records)

if df.empty:
    raise ValueError(
        "No export records found. Capture with TF_LOG=json during export "
        "or use export/hang-analysis.ipynb for hung/partial captures."
    )

print(sorted(df["type"].drop_duplicates().tolist()))

starts = (
    df[df["type"] == "export_start"]
    .copy()
    .sort_values(["resource_id", "timestamp"])
)
starts["start_timestamp"] = pd.to_datetime(starts["timestamp"], utc=True, errors="coerce")
starts["run"] = starts.groupby("resource_id").cumcount() + 1
starts = starts[
    ["resource_id", "run", "start_timestamp", "resource", "resource_type", "resource_label"]
]

ends = (
    df[df["type"] == "export_end"]
    .copy()
    .sort_values(["resource_id", "timestamp"])
)
ends["end_timestamp"] = pd.to_datetime(ends["timestamp"], utc=True, errors="coerce")
ends["run"] = ends.groupby("resource_id").cumcount() + 1
ends = ends[["resource_id", "run", "end_timestamp"]]

df_merged_export = starts.merge(ends, on=["resource_id", "run"], how="inner")
df_merged_export["time_diff_minutes"] = (
    (df_merged_export["end_timestamp"] - df_merged_export["start_timestamp"])
    .dt.total_seconds()
    / 60
)

## Type Analysis

In [ ]:
gencharts.generate_plt_by_resource_type(df, "export_start", top_n=10)
gencharts.generate_plt_by_resource_type(df, "export_end", top_n=10)

## Duration Analysis

In [ ]:
if not df_merged_export.empty:
    gencharts.generate_duration_by_resource_type(df_merged_export, metric="total", top_n=10)
    gencharts.generate_duration_by_resource_type(df_merged_export, metric="average", top_n=10)
else:
    print("No matched export_start/export_end pairs to chart.")

## Longest Running Exports

In [ ]:
if df_merged_export.empty:
    print("No matched export_start/export_end pairs to list.")
else:
    display(
        df_merged_export[
            ["resource", "resource_type", "start_timestamp", "end_timestamp", "time_diff_minutes", "run"]
        ]
        .copy()
        .sort_values(by="time_diff_minutes", ascending=False)
        .head(20)
    )

## Provider API activity

SDK DEBUG activity from the log for **completed** runs: **when** traffic spiked (timeline) and key endpoint averages. Retry/404 detail is in the exported report only. For raw HTTP pairs use `sdk-plan/sdk-analysis.ipynb`.


In [ ]:
import commonlib.prep_hang_data as hang

TAIL_MINUTES = 5
WORKFLOW = "export"
classification, _, counters = hang.load_hang_scan(c.TERRAFORM_LOG_PATH, tail_minutes=TAIL_MINUTES)
summary = hang.hang_summary_for_workflow(
    counters, TAIL_MINUTES, WORKFLOW, classification
)
df_sdk_timeline, sdk_timeline_summary, df_sdk_rates = hang.display_provider_api_activity(
    counters, summary
)
hang.display_issue_attribution(hang.build_issue_attribution(counters, summary, WORKFLOW))


## Export report

Writes `{capture-stem}-report.json` next to the log (full SDK tables including `sdk_timeline`, retry, and 404). See **`HOW-TO-READ-RESULTS.md`**.


In [ ]:
import commonlib.run_report as run_report

run_report.write_run_report(
    c.TERRAFORM_LOG_PATH,
    WORKFLOW,
    {
        **run_report.issue_attribution_bundle(counters, summary, WORKFLOW),
        "completed_exports": run_report.dataframe_records(df_merged_export),
        **run_report.sdk_report_sections(
            counters, duration_minutes=summary.get("duration_minutes")
        ),
    },
)